# LeetCode 791 — Custom Sort String bằng Python

Link đề: https://leetcode.com/problems/custom-sort-string/ · Độ khó: **Medium** · Chủ đề: Hash Table, String, Sorting

Skeleton LeetCode:

```python
class Solution:
    def customSortString(self, order: str, s: str) -> str:
        ...
```

Ghi chú: đề này có **nhiều đáp án đúng** (đề nói "trả về một hoán vị bất kỳ thỏa tính chất"), nên notebook kiểm tra bằng **hàm kiểm tra tính hợp lệ** chứ không so từng ký tự với một đáp án cứng. Ví dụ 1-2 lấy từ đề; các ví dụ khác do notebook tự đặt (đã kiểm chứng bằng code), cần thì đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho `order` (các chữ cái **khác nhau** đôi một, đã được xếp theo một thứ tự tùy chỉnh) và chuỗi `s`; hãy **hoán vị** `s` sao cho nếu `x` đứng trước `y` trong `order` thì `x` đứng trước `y` trong kết quả. Chữ cái không có trong `order` đặt ở **vị trí tùy ý**.
2. Nắm các công cụ Python cần dùng: từ điển "chữ cái -> thứ hạng", `enumerate`, `sorted(..., key=...)` và tính **ổn định** của sắp xếp, `collections.Counter`, phép nhân chuỗi `"a" * 3`, `str.join`, `functools.cmp_to_key`, `str.maketrans`/`translate`, và `itertools.permutations`.
3. Đi từ **thử mọi hoán vị** -> **sắp xếp theo khóa là thứ hạng** -> **sắp xếp bằng hàm so sánh** -> **đếm số lần xuất hiện rồi ghép theo `order`** -> **đổi nhãn bảng chữ cái rồi dùng sắp xếp mặc định**.
4. Hiểu hai mấu chốt: (a) thứ tự chỉ ràng buộc các chữ cái **có trong `order`**; mọi chữ cái khác tự do, nên kết quả **không duy nhất**; (b) thay vì sắp xếp, chỉ cần **đếm** mỗi chữ cái rồi xuất ra theo đúng thứ tự `order`.
5. Tránh các lỗi kinh điển: làm rơi các chữ cái không có trong `order`, dùng `set` làm mất chữ cái lặp, dùng `order.index(c)` cho chữ cái vắng mặt (lỗi), đếm trùng các chữ cái, và sắp xếp theo bảng chữ cái thường.

## 2. Đề bài & ví dụ

Cho hai chuỗi `order` và `s`. Mọi ký tự của `order` là **duy nhất** và đã được sắp xếp theo một thứ tự tùy chỉnh từ trước.

Hãy hoán vị các ký tự của `s` sao cho chúng khớp với thứ tự mà `order` đã sắp xếp. Cụ thể: nếu ký tự `x` xuất hiện **trước** ký tự `y` trong `order` thì `x` cũng phải xuất hiện **trước** `y` trong chuỗi kết quả.

Trả về **một hoán vị bất kỳ** của `s` thỏa tính chất trên.

**Ví dụ 1**

- Input: `order = "cba"`, `s = "abcd"`
- Output: `"cbad"`
- Giải thích: `a`, `b`, `c` xuất hiện trong `order` nên thứ tự của chúng phải là `c`, `b`, `a`. Vì `d` không có trong `order`, nó có thể ở vị trí bất kỳ: `"dcba"`, `"cdba"`, `"cbda"` cũng hợp lệ.

**Ví dụ 2**

- Input: `order = "bcafg"`, `s = "abcd"`
- Output: `"bcad"`
- Giải thích: `b`, `c`, `a` do `order` quy định thứ tự; `d` không có trong `order` nên vị trí linh hoạt. `"dbca"` hay `"bcda"` cũng hợp lệ.

**Ràng buộc:**

- `1 <= order.length <= 26`
- `1 <= s.length <= 200`
- `order` và `s` chỉ gồm chữ cái tiếng Anh viết thường.
- Mọi ký tự của `order` là **duy nhất**.

Chú ý quan trọng: `s` có thể chứa **chữ cái lặp** (ví dụ `"aabbc"`), mọi bản sao của cùng một chữ cái phải đứng **cạnh nhau** theo đúng thứ tự (trong ràng buộc "x trước y" với `x != y`, các bản sao của `x` đều phải đứng trước mọi bản sao của `y`). Một chữ cái trong `order` có thể **không xuất hiện** trong `s` (như `f`, `g` ở ví dụ 2).

In [1]:
from typing import List
from collections import Counter
from functools import cmp_to_key
from itertools import permutations
from string import ascii_lowercase
import random
import time

examples = [
    (("cba", "abcd"), "cbad"),
    (("bcafg", "abcd"), "bcad"),
]
for (order, s), sample in examples:
    print(f"order={order!r}, s={s!r}: một đáp án hợp lệ của đề là {sample!r}")

order='cba', s='abcd': một đáp án hợp lệ của đề là 'cbad'
order='bcafg', s='abcd': một đáp án hợp lệ của đề là 'bcad'


## 3. Kiến thức nền

### 3.1. Vì sao có nhiều đáp án đúng, và cách kiểm tra

Chỉ các chữ cái **có trong `order`** bị ràng buộc thứ tự. Chữ cái khác tự do. Nên ta không thể chỉ so sánh kết quả với một chuỗi cố định; ta cần **hàm kiểm tra tính hợp lệ** `is_valid(order, s, result)`:

1. `result` là **hoán vị** của `s` (cùng các ký tự, cùng số lần xuất hiện): `sorted(result) == sorted(s)`.
2. Lấy các ký tự của `result` **có trong `order`**, đổi thành **thứ hạng** (vị trí trong `order`); dãy thứ hạng này phải **không giảm**.

Dãy thứ hạng "không giảm" (cho phép bằng nhau) vì các bản sao của cùng một chữ cái có cùng thứ hạng và đứng cạnh nhau.

In [2]:
def is_valid(order, s, result):
    if sorted(result) != sorted(s):                  # phải là hoán vị của s
        return False
    rank = {c: i for i, c in enumerate(order)}
    seq = [rank[c] for c in result if c in rank]     # chỉ xét chữ cái có trong order
    return all(seq[i] <= seq[i + 1] for i in range(len(seq) - 1))

print(is_valid("cba", "abcd", "cbad"), is_valid("cba", "abcd", "dcba"), is_valid("cba", "abcd", "cdba"))
print(is_valid("cba", "abcd", "abcd"), "<- sai thứ tự")
print(is_valid("cba", "abcd", "cba"), "<- thiếu d, không phải hoán vị")

True True True
False <- sai thứ tự
False <- thiếu d, không phải hoán vị


### 3.2. Từ điển thứ hạng: `rank[c]`

`dict` ánh xạ **khóa -> giá trị**. Ta tạo `rank` để tra "chữ cái `c` đứng thứ mấy trong `order`" trong `O(1)`: `rank = {c: i for i, c in enumerate(order)}`. `enumerate` cho từng cặp `(chỉ số, phần tử)`. Chữ cái vắng mặt: dùng `rank.get(c, default)` để cho thứ hạng mặc định thay vì lỗi `KeyError`.

In [3]:
order = "cba"
rank = {c: i for i, c in enumerate(order)}
print(rank)
print(rank["b"], rank.get("d", 99))
try:
    rank["d"]
except KeyError as e:
    print("rank['d'] -> KeyError:", e)

{'c': 0, 'b': 1, 'a': 2}
1 99
rank['d'] -> KeyError: 'd'


### 3.3. `sorted(..., key=...)` và tính ổn định

`sorted(iterable, key=f)` sắp xếp theo giá trị `f(phần tử)`. Sắp xếp của Python là **ổn định (stable)**: hai phần tử có cùng khóa **giữ nguyên thứ tự ban đầu**. Dùng được để đặt các chữ cái vắng mặt ở cuối: cho chúng khóa lớn hơn mọi thứ hạng.

In [4]:
s = "abcdda"
print(sorted(s, key=lambda c: rank.get(c, 99)))
print(sorted("dxdyd", key=lambda c: 0))              # cùng khóa -> giữ nguyên thứ tự
print("".join(sorted("abcd", key=lambda c: rank.get(c, 99))))

['c', 'b', 'a', 'a', 'd', 'd']
['d', 'x', 'd', 'y', 'd']
cbad


### 3.4. `Counter` và phép nhân chuỗi

`Counter(s)` đếm số lần mỗi ký tự xuất hiện: `Counter("abca")` = `{'a': 2, 'b': 1, 'c': 1}`. Truy cập chữ cái vắng mặt cho `0` (không lỗi). `"a" * 3` ghép ba chữ `a`: `"aaa"`; `"a" * 0` là chuỗi rỗng. `"".join(list)` ghép các chuỗi.

In [5]:
cnt = Counter("abcdda")
print(cnt, cnt["a"], cnt["z"])
print("a" * 3, repr("a" * 0), "".join(["ab", "c" * 2, "d"]))
print(cnt.most_common(2))

Counter({'a': 2, 'd': 2, 'b': 1, 'c': 1}) 2 0
aaa '' abccd
[('a', 2), ('d', 2)]


### 3.5. `cmp_to_key`: sắp xếp bằng hàm so sánh

Hàm so sánh `cmp(a, b)` trả số âm nếu `a` đứng trước `b`, dương nếu sau, `0` nếu ngang nhau. `cmp_to_key(cmp)` biến nó thành `key` cho `sorted`. Cách này cho phép tùy biến luật so sánh phức tạp.

In [6]:
def cmp(a, b):
    return rank.get(a, 99) - rank.get(b, 99)

print(sorted("abcd", key=cmp_to_key(cmp)))
print(cmp("c", "a"), cmp("a", "c"), cmp("a", "a"), cmp("d", "d"))

['c', 'b', 'a', 'd']
-2 2 0 0


### 3.6. `str.maketrans` và `translate`: thay chữ cái hàng loạt

`str.maketrans(src, dst)` tạo bảng đổi `src[i] -> dst[i]`; `s.translate(bảng)` áp dụng cho cả chuỗi. Ta dùng nó để **đổi nhãn bảng chữ cái** (cách 5): đưa thứ tự tùy chỉnh về thứ tự `a, b, c, ...` thông thường để dùng `sorted` mặc định rồi đổi ngược.

In [7]:
fwd = str.maketrans("cba", "abc")
print("abcab".translate(fwd))
print(sorted("abcab".translate(fwd)))
back = str.maketrans("abc", "cba")
print("".join(sorted("abcab".translate(fwd))).translate(back))

cbacb
['a', 'b', 'b', 'c', 'c']
cbbaa


### 3.7. `itertools.permutations`: mọi hoán vị (cho cách vét cạn)

`permutations(s)` sinh mọi hoán vị (theo vị trí; chữ cái lặp cho nhiều hoán vị trùng chuỗi). Chuỗi dài `n` có `n!` hoán vị: `7! = 5040`, `10! = 3.628.800`: chỉ dùng được cho chuỗi rất ngắn. Dùng `set(...)` để bỏ trùng.

In [8]:
perms = list(permutations("abc"))
print(len(perms), perms[:3])
print(len(set(permutations("aab"))), "hoán vị khác nhau của 'aab' trong tổng", len(list(permutations("aab"))))

def all_valid(order, s):
    """Mọi đáp án hợp lệ (khác nhau) bằng vét cạn hoán vị."""
    return sorted(x for x in {"".join(p) for p in permutations(s)} if is_valid(order, s, x))

print(all_valid("cba", "abcd"))

6 [('a', 'b', 'c'), ('a', 'c', 'b'), ('b', 'a', 'c')]
3 hoán vị khác nhau của 'aab' trong tổng 6
['cbad', 'cbda', 'cdba', 'dcba']


Chỉ có **4** đáp án hợp lệ cho `order = "cba"`, `s = "abcd"` (chữ `d` chọn một trong 4 vị trí): khớp mô tả trong đề. Hàm `all_valid` ở trên (liệt kê mọi hoán vị rồi lọc) sẽ làm **cơ sở kiểm chứng độc lập** cho các chuỗi ngắn.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với `order = "cba"`, `s = "abcd"`:

1. Chữ cái `c` đứng **đầu tiên** trong `order` -> mọi chữ `c` của `s` phải đứng trước mọi chữ `b`, và `b` đứng trước mọi chữ `a`.
2. Trong `s` có: `a` một lần, `b` một lần, `c` một lần, `d` một lần.
3. Viết lần lượt theo `order`: "`c` một lần" + "`b` một lần" + "`a` một lần" = `"cba"`.
4. Còn chữ `d` (không có trong `order`): tùy ý, ta thêm vào cuối -> `"cbad"`.

Bây giờ thử với chữ cái **lặp**: `order = "kqep"`, `s = "pekeq"`. Đếm: `p` 1, `e` 2, `k` 1, `q` 1. Theo thứ tự `order`: `k` (1 lần) `q` (1 lần) `e` (2 lần) `p` (1 lần) -> `"kqeep"`. Không còn chữ nào khác.

| Bước | Chữ cái trong `order` | Số lần trong `s` | Phần ghép | Kết quả tạm |
|---|---|---|---|---|
| 1 | `k` | 1 | `k` | `k` |
| 2 | `q` | 1 | `q` | `kq` |
| 3 | `e` | 2 | `ee` | `kqee` |
| 4 | `p` | 1 | `p` | `kqeep` |
| 5 | còn lại (không có) | - | - | `kqeep` |

Nhận xét: ta **không cần thực sự "sắp xếp"** gì cả; chỉ cần đếm rồi xuất theo `order`. Phần còn lại (chữ cái không có trong `order`) đặt ở đâu cũng được; đặt ở cuối là cách đơn giản nhất.

## 5. Cách 1 — Brute Force / baseline: thử mọi hoán vị và kiểm tra

**Ý tưởng.** Với mỗi hoán vị của `s`, kiểm tra bằng `is_valid`; trả về hoán vị hợp lệ đầu tiên. Luôn có ít nhất một hoán vị hợp lệ nên chắc chắn dừng. Cách này **đúng nhưng cực chậm** (`n!`), chỉ dùng được cho `s` rất ngắn.

In [9]:
def custom_sort_brute(order, s):
    for p in permutations(s):
        cand = "".join(p)
        if is_valid(order, s, cand):
            return cand

for (order, s), sample in examples:
    got = custom_sort_brute(order, s)
    print(order, s, "->", got, "hợp lệ?", is_valid(order, s, got))

cba abcd -> cbad hợp lệ? True
bcafg abcd -> bcad hợp lệ? True


In [10]:
class Solution1:
    """Cách 1: thử mọi hoán vị (chỉ cho chuỗi rất ngắn)."""
    def customSortString(self, order: str, s: str) -> str:
        rank = {c: i for i, c in enumerate(order)}
        for p in permutations(s):
            seq = [rank[c] for c in p if c in rank]
            if all(seq[i] <= seq[i + 1] for i in range(len(seq) - 1)):
                return "".join(p)

for (order, s), _ in examples:
    assert is_valid(order, s, Solution1().customSortString(order, s))
print("cách 1 qua các ví dụ (kiểm tra bằng is_valid)")

cách 1 qua các ví dụ (kiểm tra bằng is_valid)


**Độ phức tạp.** Có `n!` hoán vị, mỗi hoán vị kiểm tra `O(n)`: **Time `O(n! * n)`**, Space `O(n)`. Với `n = 200` là bất khả thi.

**Nút thắt.** Ta không cần thử hoán vị nào cả: tính chất cần có chỉ phụ thuộc **thứ hạng** của từng chữ cái. Đo thử để thấy độ chậm tăng vọt:

In [11]:
print("n  | thời gian brute (ms) | trường hợp xấu: các hoán vị hợp lệ nằm ở cuối")
for n in [5, 6, 7, 8]:
    order = ascii_lowercase[:n][::-1]            # thứ tự ngược: chuỗi xuôi gần như xấu nhất
    s = ascii_lowercase[:n]
    t0 = time.perf_counter()
    r = custom_sort_brute(order, s)
    print(f"{n:>2} | {(time.perf_counter() - t0) * 1000:>14.1f}      -> {r}")

n  | thời gian brute (ms) | trường hợp xấu: các hoán vị hợp lệ nằm ở cuối
 5 |            0.2      -> edcba
 6 |            1.2      -> fedcba
 7 |            9.5      -> gfedcba
 8 |           87.2      -> hgfedcba


Thời gian tăng khoảng `n` lần mỗi khi thêm một chữ cái (do `n!`). Cần một cách trực tiếp.

## 6. Ý tưởng thuật toán / bước đột phá

Ràng buộc "`x` trước `y` nếu `x` trước `y` trong `order`" chỉ phụ thuộc **thứ hạng** của chữ cái. Nếu gán mỗi chữ cái một **số thứ hạng** (vị trí trong `order`; chữ cái vắng mặt cho thứ hạng lớn nhất) thì bài toán trở thành: **sắp xếp `s` theo khóa là thứ hạng**. Từ đây có ba cách nghĩ:

- **Sắp xếp theo khóa** (cách 2) hoặc **theo hàm so sánh** (cách 3): `O(n log n)`.
- **Đếm rồi ghép** (cách 4): vì bảng chữ cái chỉ có `26` chữ, ta không cần sắp xếp; đếm số lần mỗi chữ cái, rồi xuất `c * count[c]` theo đúng thứ tự `order`, sau đó thêm các chữ cái còn lại: **`O(n)`**.
- **Đổi nhãn bảng chữ cái** (cách 5): ánh xạ thứ tự tùy chỉnh về `a, b, c, ...`, rồi `sorted` mặc định, rồi ánh xạ ngược.

**Bước đột phá (một câu):** *thứ tự chỉ nằm ở thứ hạng của từng chữ cái, nên đếm và xuất theo `order` là đủ; chữ cái ngoài `order` đặt ở đâu cũng đúng.*

## 7. Các cách tiếp theo

### Cách 2 — Sắp xếp theo khóa (`sorted` với `key = thứ hạng`)

**Giải thích từng đoạn.**

1. `rank = {c: i for i, c in enumerate(order)}`.
2. Khóa của chữ cái `c`: `rank.get(c, len(order))`; chữ cái vắng mặt có khóa `len(order)` lớn hơn mọi thứ hạng thật -> bị đẩy xuống **cuối**. (Nếu cho khóa `-1` chúng lên đầu: cũng hợp lệ.)
3. `"".join(sorted(s, key=...))`.

Các chữ cái cùng khóa (cùng chữ, hoặc cùng vắng mặt) giữ thứ tự ban đầu vì `sorted` ổn định.

In [12]:
order, s = "cba", "abcdda"
rank = {c: i for i, c in enumerate(order)}
key = lambda c: rank.get(c, len(order))
print([(c, key(c)) for c in s])
print("".join(sorted(s, key=key)))

[('a', 2), ('b', 1), ('c', 0), ('d', 3), ('d', 3), ('a', 2)]
cbaadd


**Code nộp LeetCode (cách 2):**

In [13]:
class Solution2:
    """Cách 2: sorted với khóa là thứ hạng trong order."""
    def customSortString(self, order: str, s: str) -> str:
        rank = {c: i for i, c in enumerate(order)}
        return "".join(sorted(s, key=lambda c: rank.get(c, len(order))))

for (order, s), _ in examples:
    assert is_valid(order, s, Solution2().customSortString(order, s))
print("cách 2 qua các ví dụ")

cách 2 qua các ví dụ


**Độ phức tạp (cách 2).** Sắp xếp `n` ký tự: **Time `O(n log n)`**, **Space `O(n)`** (danh sách tạm + từ điển 26 mục).

### Cách 3 — Sắp xếp bằng hàm so sánh (`cmp_to_key`)

**Giải thích.** Giống cách 2 nhưng ta viết rõ luật so sánh hai chữ cái: chữ nào có thứ hạng **nhỏ hơn** thì đứng trước. Hàm `cmp(a, b)` trả về `rank(a) - rank(b)`; `cmp_to_key` chuyển nó thành khóa. Hữu ích khi luật so sánh không viết gọn được thành một khóa số (ở đây thì được, nhưng cách này giúp bạn làm quen với "so sánh tùy biến").

In [14]:
order, s = "bcafg", "abcd"
rank = {c: i for i, c in enumerate(order)}

def cmp(a, b):
    ra, rb = rank.get(a, len(order)), rank.get(b, len(order))
    return ra - rb                      # âm: a đứng trước b

print(sorted(s, key=cmp_to_key(cmp)))
print([cmp(x, y) for x, y in [("b", "c"), ("c", "b"), ("a", "d"), ("d", "d")]])

['b', 'c', 'a', 'd']
[-1, 1, -3, 0]


**Code nộp LeetCode (cách 3):**

In [15]:
class Solution3:
    """Cách 3: sorted với hàm so sánh."""
    def customSortString(self, order: str, s: str) -> str:
        rank = {c: i for i, c in enumerate(order)}
        n = len(order)

        def cmp(a, b):
            return rank.get(a, n) - rank.get(b, n)

        return "".join(sorted(s, key=cmp_to_key(cmp)))

for (order, s), _ in examples:
    assert is_valid(order, s, Solution3().customSortString(order, s))
print("cách 3 qua các ví dụ")

cách 3 qua các ví dụ


**Độ phức tạp (cách 3).** `O(n log n)` so sánh, mỗi so sánh `O(1)`; chậm hơn cách 2 một chút vì gọi hàm Python mỗi lần so sánh. **Space `O(n)`**.

### Cách 4 — Đếm số lần xuất hiện rồi ghép theo `order` (khuyên dùng)

**Giải thích từng đoạn.**

1. `cnt = Counter(s)`: số lần mỗi chữ cái trong `s`.
2. Với mỗi chữ cái `c` **theo thứ tự `order`**: nếu `cnt[c] > 0` thì thêm `c * cnt[c]` vào kết quả và **xóa** `c` khỏi `cnt` (đã dùng xong).
3. Các chữ cái còn lại trong `cnt` (không có trong `order`) thêm ở cuối theo cách nào cũng được, ví dụ `c * cnt[c]`.

Không có sắp xếp: chỉ vài vòng lặp qua `order` (tối đa 26) và qua các chữ cái còn lại.

In [16]:
def demo_counting(order, s):
    cnt = Counter(s)
    print("đếm:", dict(cnt))
    parts = []
    for c in order:
        if c in cnt:
            parts.append(c * cnt[c])
            print(f"  gặp {c!r} trong order: thêm {c * cnt[c]!r}")
            del cnt[c]
    for c, k in cnt.items():
        parts.append(c * k)
        print(f"  còn lại {c!r} (không có trong order): thêm {c * k!r}")
    return "".join(parts)

print("kết quả:", demo_counting("cba", "abcd"))
print("kết quả:", demo_counting("kqep", "pekeq"))

đếm: {'a': 1, 'b': 1, 'c': 1, 'd': 1}
  gặp 'c' trong order: thêm 'c'
  gặp 'b' trong order: thêm 'b'
  gặp 'a' trong order: thêm 'a'
  còn lại 'd' (không có trong order): thêm 'd'
kết quả: cbad
đếm: {'p': 1, 'e': 2, 'k': 1, 'q': 1}
  gặp 'k' trong order: thêm 'k'
  gặp 'q' trong order: thêm 'q'
  gặp 'e' trong order: thêm 'ee'
  gặp 'p' trong order: thêm 'p'
kết quả: kqeep


**Code nộp LeetCode (cách 4) - cũng là code chính ở mục 9:**

In [17]:
class Solution4:
    """Cách 4: đếm số lần xuất hiện rồi ghép theo thứ tự order."""
    def customSortString(self, order: str, s: str) -> str:
        cnt = Counter(s)
        res = []
        for c in order:
            if c in cnt:
                res.append(c * cnt.pop(c))       # lấy hết các chữ c, rồi bỏ khỏi bộ đếm
        for c, k in cnt.items():                 # chữ cái không có trong order
            res.append(c * k)
        return "".join(res)

for (order, s), _ in examples:
    assert is_valid(order, s, Solution4().customSortString(order, s))
print("cách 4 qua các ví dụ")

cách 4 qua các ví dụ


**Độ phức tạp (cách 4).** Đếm `O(n)`, duyệt `order` tối đa `26` bước, ghép chuỗi `O(n)`: **Time `O(n)`**, **Space `O(n)`** cho kết quả (phụ `O(26)`). Là cách tốt nhất về mặt tiệm cận.

### Cách 5 — Đổi nhãn bảng chữ cái rồi dùng `sorted` mặc định

**Giải thích.**

- Muốn dùng `sorted` thông thường (theo `a < b < c < ...`) nên ta **đổi tên** các chữ cái: chữ thứ `i` của `order` đổi thành chữ cái thứ `i` của bảng `a..z`.
- Các chữ cái có trong `s` nhưng **không** có trong `order` đổi thành những chữ cái **tiếp theo** (các nhãn còn trống), nên đứng sau mọi chữ cái của `order`.
- Hai danh sách `src` (tên cũ) và `dst` (tên mới) có cùng độ dài ≤ 26 nên là **song ánh**: đổi tiến rồi đổi ngược cho đúng chuỗi cũ.
- `sorted(s.translate(tiến))` -> ghép -> `translate(ngược)`.

In [18]:
def demo_relabel(order, s):
    others = sorted(set(s) - set(order))
    src = order + "".join(others)
    dst = ascii_lowercase[:len(src)]
    fwd = str.maketrans(src, dst)
    back = str.maketrans(dst, src)
    relabeled = s.translate(fwd)
    print(f"src={src!r} -> dst={dst!r}")
    print("s sau khi đổi nhãn :", relabeled)
    sorted_str = "".join(sorted(relabeled))
    print("sau sorted mặc định :", sorted_str)
    return sorted_str.translate(back)

print("kết quả:", demo_relabel("cba", "abcd"))
print("kết quả:", demo_relabel("bcafg", "abcdxd"))

src='cbad' -> dst='abcd'
s sau khi đổi nhãn : cbad
sau sorted mặc định : abcd
kết quả: cbad
src='bcafgdx' -> dst='abcdefg'
s sau khi đổi nhãn : cabfgf
sau sorted mặc định : abcffg
kết quả: bcaddx


**Code nộp LeetCode (cách 5):**

In [19]:
class Solution5:
    """Cách 5: đổi nhãn bảng chữ cái, sorted mặc định, đổi ngược."""
    def customSortString(self, order: str, s: str) -> str:
        others = sorted(set(s) - set(order))
        src = order + "".join(others)                 # các chữ cái đôi một khác nhau, tối đa 26
        dst = ascii_lowercase[:len(src)]
        fwd = str.maketrans(src, dst)
        back = str.maketrans(dst, src)
        return "".join(sorted(s.translate(fwd))).translate(back)

for (order, s), _ in examples:
    assert is_valid(order, s, Solution5().customSortString(order, s))
print("cách 5 qua các ví dụ")

cách 5 qua các ví dụ


**Độ phức tạp (cách 5).** `sorted` `O(n log n)`, các `translate` `O(n)`: **Time `O(n log n)`**, **Space `O(n)`**. Hơi "vòng vo" nhưng minh họa kỹ thuật **đổi nhãn** (relabeling) hữu ích ở nhiều bài (ví dụ Verifying an Alien Dictionary).

**Tóm tắt năm cách:** (1) thử hoán vị `O(n!)`; (2) `sorted` theo khóa `O(n log n)`; (3) `sorted` theo hàm so sánh `O(n log n)`; (4) đếm rồi ghép `O(n)`; (5) đổi nhãn rồi `sorted` mặc định `O(n log n)`.

## 8. TRACE từng bước

### 8.1. Trace cách 4 (đếm rồi ghép) trên `order = "bcafg"`, `s = "abcdd"`

| Cột | Ý nghĩa |
|---|---|
| `c` | chữ cái đang xét theo thứ tự `order` |
| `cnt[c]` | số lần `c` còn trong bộ đếm |
| `thêm` | phần ghép vào kết quả |
| `kết quả` | chuỗi kết quả tạm |

In [20]:
def trace_counting(order, s):
    cnt = Counter(s)
    res = ""
    print("bộ đếm ban đầu:", dict(cnt))
    for c in order:
        k = cnt.get(c, 0)
        if k:
            res += c * k
            del cnt[c]
            print(f"c={c!r}: cnt={k} -> thêm {c * k!r} | kết quả = {res!r} | còn lại {dict(cnt)}")
        else:
            print(f"c={c!r}: không có trong s -> bỏ qua")
    for c, k in list(cnt.items()):
        res += c * k
        print(f"còn lại {c!r} x{k} -> thêm {c * k!r} | kết quả = {res!r}")
    return res

out = trace_counting("bcafg", "abcdd")
print("kết quả:", out, "| hợp lệ?", is_valid("bcafg", "abcdd", out))

bộ đếm ban đầu: {'a': 1, 'b': 1, 'c': 1, 'd': 2}
c='b': cnt=1 -> thêm 'b' | kết quả = 'b' | còn lại {'a': 1, 'c': 1, 'd': 2}
c='c': cnt=1 -> thêm 'c' | kết quả = 'bc' | còn lại {'a': 1, 'd': 2}
c='a': cnt=1 -> thêm 'a' | kết quả = 'bca' | còn lại {'d': 2}
c='f': không có trong s -> bỏ qua
c='g': không có trong s -> bỏ qua
còn lại 'd' x2 -> thêm 'dd' | kết quả = 'bcadd'
kết quả: bcadd | hợp lệ? True


Chú ý `f` và `g` có trong `order` nhưng **không** có trong `s`, nên bị bỏ qua. `d` (hai lần) không có trong `order` nên được thêm cuối cùng: `"bcadd"`.

### 8.2. Trace cách 2 (sắp xếp theo khóa)

In khóa của từng ký tự rồi kết quả sau sắp xếp (ổn định).

In [21]:
def trace_sort_key(order, s):
    rank = {c: i for i, c in enumerate(order)}
    n = len(order)
    keyed = [(c, rank.get(c, n)) for c in s]
    print("(ký tự, khóa) =", keyed)
    ordered = sorted(keyed, key=lambda t: t[1])
    print("sau khi sắp theo khóa:", ordered)
    return "".join(c for c, _ in ordered)

out = trace_sort_key("cba", "abdcad")
print("kết quả:", out, "| hợp lệ?", is_valid("cba", "abdcad", out))

(ký tự, khóa) = [('a', 2), ('b', 1), ('d', 3), ('c', 0), ('a', 2), ('d', 3)]
sau khi sắp theo khóa: [('c', 0), ('b', 1), ('a', 2), ('a', 2), ('d', 3), ('d', 3)]
kết quả: cbaadd | hợp lệ? True


Hai chữ `d` có cùng khóa `3` (vắng mặt): chúng đứng ở cuối và giữ thứ tự gốc. Các chữ `c`, `b`, `a` có khóa `0, 1, 2` nên lần lượt đứng đầu.

### 8.3. Trace cách 5 (đổi nhãn)

In [22]:
def trace_relabel(order, s):
    others = sorted(set(s) - set(order))
    src = order + "".join(others)
    dst = ascii_lowercase[:len(src)]
    print("bảng đổi tiến :", dict(zip(src, dst)))
    relabeled = s.translate(str.maketrans(src, dst))
    print(f"{s!r} -> đổi nhãn -> {relabeled!r}")
    sorted_str = "".join(sorted(relabeled))
    print(f"sorted mặc định -> {sorted_str!r}")
    result = sorted_str.translate(str.maketrans(dst, src))
    print(f"đổi ngược -> {result!r}")
    return result

out = trace_relabel("cba", "abdcad")
print("hợp lệ?", is_valid("cba", "abdcad", out))

bảng đổi tiến : {'c': 'a', 'b': 'b', 'a': 'c', 'd': 'd'}
'abdcad' -> đổi nhãn -> 'cbdacd'
sorted mặc định -> 'abccdd'
đổi ngược -> 'cbaadd'
hợp lệ? True


## 9. Code nộp LeetCode

Dùng **cách 4** (đếm rồi ghép):

```python
class Solution:
    def customSortString(self, order: str, s: str) -> str:
        cnt = Counter(s)
        res = []
        for c in order:
            if c in cnt:
                res.append(c * cnt.pop(c))       # lấy hết các chữ c, rồi bỏ khỏi bộ đếm
        for c, k in cnt.items():                 # chữ cái không có trong order
            res.append(c * k)
        return "".join(res)
```

(Trên LeetCode `Counter` cần `from collections import Counter`, thường đã có sẵn.)

Chạy lại đúng đoạn code trên:

In [23]:
class Solution:
    def customSortString(self, order: str, s: str) -> str:
        cnt = Counter(s)
        res = []
        for c in order:
            if c in cnt:
                res.append(c * cnt.pop(c))       # lấy hết các chữ c, rồi bỏ khỏi bộ đếm
        for c, k in cnt.items():                 # chữ cái không có trong order
            res.append(c * k)
        return "".join(res)

# Mô phỏng cách LeetCode gọi: có nhiều đáp án đúng nên kiểm tra bằng is_valid
for (order, s), sample in examples:
    got = Solution().customSortString(order, s)
    print(repr(order), repr(s), "->", repr(got), "| hợp lệ?", is_valid(order, s, got))
    assert is_valid(order, s, got)

'cba' 'abcd' -> 'cbad' | hợp lệ? True
'bcafg' 'abcd' -> 'bcad' | hợp lệ? True


## 10. Vì sao thuật toán đúng

**Cần chứng minh** kết quả `R` của cách 4 (a) là hoán vị của `s`, (b) thỏa ràng buộc thứ tự.

**(a) Hoán vị.** `Counter(s)` ghi đúng số lần mỗi chữ cái. Mỗi chữ cái `c` được xuất ra đúng `cnt[c]` lần: hoặc trong vòng lặp qua `order` (nếu `c` thuộc `order`, rồi bị `pop` nên không bị xuất lần nữa), hoặc trong vòng lặp phần còn lại (nếu `c` không thuộc `order`, vì khi đó nó chưa bị `pop`). Do đó mỗi chữ cái xuất hiện đúng bằng số lần của nó trong `s`, mọi chữ cái của `s` đều có mặt: `R` là hoán vị của `s`.

**(b) Thứ tự.** Lấy hai chữ cái khác nhau `x`, `y` đều thuộc `order` sao cho `x` đứng trước `y` trong `order`. Vòng lặp qua `order` xử lý `x` **trước** `y` và xuất **toàn bộ** các bản sao của `x` ngay lúc đó, rồi sau đó mới xuất các bản sao của `y`. Vậy mọi `x` trong `R` đứng trước mọi `y`. Các chữ cái ngoài `order` không bị ràng buộc nên vị trí cuối cùng là hợp lệ.

**Cách 2, 3, 5** sắp xếp theo thứ hạng (khóa không giảm): mọi ký tự có thứ hạng nhỏ hơn đứng trước; hai chữ cùng thứ hạng là bản sao của cùng chữ cái (hoặc cùng vắng mặt) nên không bị ràng buộc. Cách 5 thêm: phép đổi nhãn là **song ánh** giữ nguyên thứ tự tương đối theo `order` (chữ thứ `i` của `order` nhận nhãn thứ `i`), nên sắp xếp mặc định sau khi đổi nhãn đúng bằng sắp xếp theo `order` trước khi đổi.

**Cách 1** thử mọi hoán vị và kiểm tra trực tiếp nên đúng theo định nghĩa; luôn tồn tại ít nhất một hoán vị hợp lệ (chính kết quả của các cách trên) nên không trả về `None`.

**Về hàm kiểm tra `is_valid`:** "dãy thứ hạng của các chữ cái thuộc `order` không giảm" tương đương đúng với yêu cầu "nếu `x` trước `y` trong `order` thì mọi `x` đứng trước mọi `y` trong kết quả" (hai chữ cái khác nhau, thứ hạng khác nhau). Đó là lý do ta dùng nó làm thước đo duy nhất.

## 11. Phân tích Time / Space Complexity

`n = len(s)` (tối đa `200`), `m = len(order)` (tối đa `26`).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Thử mọi hoán vị | `O(n! * n)` | `O(n)` | chỉ cho `n` rất nhỏ |
| 2. `sorted` theo khóa | `O(n log n)` | `O(n)` | ngắn gọn |
| 3. `sorted` theo hàm so sánh | `O(n log n)` | `O(n)` | hàm gọi lại nhiều lần nên chậm hơn |
| 4. Đếm rồi ghép | `O(n + m)` | `O(n)` | **dùng để nộp** |
| 5. Đổi nhãn + `sorted` | `O(n log n)` | `O(n)` | kỹ thuật relabeling |

Với `n <= 200` mọi cách (trừ cách 1) chạy tức thì; chọn cách 4 vì tiệm cận tốt nhất và dễ chứng minh.

## 12. Bộ test

### 12.1. Ví dụ của đề và cách kiểm tra khi có nhiều đáp án

Mỗi cách có thể trả về một hoán vị hợp lệ **khác nhau**. Ta kiểm tra bằng `is_valid` (mục 3.1) và thêm một **tính chất mạnh hơn** không phụ thuộc vị trí của chữ cái ngoài `order`:

- Chiếu kết quả lên các chữ cái trong `order` (bỏ các chữ cái khác): chuỗi thu được là **duy nhất**, bằng `"".join(c * Counter(s)[c] for c in order)`.
- Các chữ cái ngoài `order` (đa tập) đúng bằng đa tập tương ứng của `s`.

In [24]:
METHODS = {
    "1-hoan-vi": Solution1().customSortString,
    "2-sort-key": Solution2().customSortString,
    "3-cmp": Solution3().customSortString,
    "4-dem": Solution4().customSortString,
    "5-doi-nhan": Solution5().customSortString,
    "nop": Solution().customSortString,
}

def check_property(order, s, result):
    assert is_valid(order, s, result), ("không hợp lệ", order, s, result)
    cs = Counter(s)
    projected = "".join(c for c in result if c in order)
    assert projected == "".join(c * cs[c] for c in order), ("chiếu sai", order, s, result)
    others_res = sorted(c for c in result if c not in order)
    others_s = sorted(c for c in s if c not in order)
    assert others_res == others_s

def check_all(order, s, skip=()):
    for name, fn in METHODS.items():
        if name in skip:
            continue
        check_property(order, s, fn(order, s))

for (order, s), sample in examples:
    check_property(order, s, sample)                      # đáp án mẫu của đề cũng phải hợp lệ
    check_all(order, s)
for res in ["cbad", "dcba", "cdba", "cbda"]:
    assert is_valid("cba", "abcd", res)                  # bốn đáp án hợp lệ theo giải thích của đề
print("ví dụ của đề: OK")

ví dụ của đề: OK


### 12.2. Edge case

- `s` chỉ có chữ cái **không** thuộc `order` (mọi thứ tự đều hợp lệ).
- `s` chỉ có chữ cái thuộc `order`, **lặp nhiều lần**, đã đúng thứ tự hoặc ngược thứ tự.
- `order` có chữ cái **không** xuất hiện trong `s`.
- `order` chỉ một chữ cái; `s` một chữ cái.
- `s` toàn một chữ cái lặp.
- `order` đủ 26 chữ cái, `s` là hoán vị bảng chữ cái.

In [25]:
edge = [
    ("a", "a"),
    ("a", "b"),
    ("b", "abc"),
    ("abc", "xyz"),                     # không chữ cái nào thuộc order
    ("abc", "cba"),
    ("cba", "abc"),
    ("cba", "aabbcc"),
    ("cba", "ccbbaa"),
    ("kqep", "pekeq"),
    ("zyx", "xyzxyzxyz"),
    ("bcafg", "abcd"),
    ("abcdefghijklmnopqrstuvwxyz"[::-1], "abcdefghijklmnopqrstuvwxyz"),
    ("q", "aaaaaaaaaq"),
    ("xyz", "aaaaa"),
]
for order, s in edge:
    skip = ("1-hoan-vi",) if len(s) > 7 else ()
    check_all(order, s, skip=skip)
    print(f"order={order[:6]!r:9} s={s[:14]!r:16} -> {Solution().customSortString(order, s)[:30]!r}")
print("edge case: OK")

order='a'       s='a'              -> 'a'
order='a'       s='b'              -> 'b'
order='b'       s='abc'            -> 'bac'
order='abc'     s='xyz'            -> 'xyz'
order='abc'     s='cba'            -> 'abc'
order='cba'     s='abc'            -> 'cba'
order='cba'     s='aabbcc'         -> 'ccbbaa'
order='cba'     s='ccbbaa'         -> 'ccbbaa'
order='kqep'    s='pekeq'          -> 'kqeep'
order='zyx'     s='xyzxyzxyz'      -> 'zzzyyyxxx'
order='bcafg'   s='abcd'           -> 'bcad'
order='zyxwvu'  s='abcdefghijklmn' -> 'zyxwvutsrqponmlkjihgfedcba'
order='q'       s='aaaaaaaaaq'     -> 'qaaaaaaaaa'
order='xyz'     s='aaaaa'          -> 'aaaaa'
edge case: OK


### 12.3. Phản ví dụ: input làm lộ các lỗi quen thuộc

- `s` có **chữ cái lặp** và chữ cái **ngoài `order`** (bắt lỗi dùng `set`, làm rơi chữ cái, đếm trùng).
- `order` chứa chữ cái **không có trong `s`** (bắt lỗi `order.index`... hay phép nhân với `0`).
- Thứ tự `order` **ngược** bảng chữ cái (bắt lỗi dùng `sorted(s)` thường).

In [26]:
tricky = [
    ("cba", "abdcadd"),
    ("hgfedcba", "aabbccdd"),
    ("zxy", "yyxxzzaab"),
    ("bca", "aaabbbcccddd"),
    ("mnop", "pomnxxpom"),
]
for order, s in tricky:
    check_all(order, s, skip=("1-hoan-vi",))
    print(f"{order:9} {s:14} -> {Solution().customSortString(order, s)}")

cba       abdcadd        -> cbaaddd
hgfedcba  aabbccdd       -> ddccbbaa
zxy       yyxxzzaab      -> zzxxyyaab
bca       aaabbbcccddd   -> bbbcccaaaddd
mnop      pomnxxpom      -> mmnooppxx


### 12.4. Đối chiếu độc lập với bộ liệt kê mọi hoán vị (chuỗi ngắn)

Với `s` ngắn (`<= 6` ký tự), `all_valid(order, s)` liệt kê **mọi đáp án hợp lệ** bằng vét cạn: đáp án của mỗi cách phải **thuộc** tập này. Đây là kiểm chứng độc lập vì `all_valid` chỉ dùng `permutations` và `is_valid`.

In [27]:
rng = random.Random(791)
count = 0
for _ in range(400):
    alphabet = rng.choice(["abc", "abcd", "abcdef"])
    order = "".join(rng.sample(alphabet, rng.randint(1, len(alphabet))))
    s = "".join(rng.choice(alphabet) for _ in range(rng.randint(1, 6)))
    valid_set = set(all_valid(order, s))
    assert valid_set                                         # luôn có ít nhất một đáp án
    for name, fn in METHODS.items():
        assert fn(order, s) in valid_set, (name, order, s)
    count += 1
print("đã đối chiếu", count, "input ngẫu nhiên với tập đáp án hợp lệ do vét cạn sinh ra: OK")

đã đối chiếu 400 input ngẫu nhiên với tập đáp án hợp lệ do vét cạn sinh ra: OK


### 12.5. Ngẫu nhiên đúng ràng buộc (seed cố định) và tính chất thống kê

Sinh `order` (một hoán vị ngẫu nhiên của một tập con chữ cái, độ dài `1..26`) và `s` (độ dài `1..200`, chữ cái thường). Kiểm tra `check_property` cho mọi cách (trừ cách 1 vì hàm giai thừa), và kiểm tra thêm **số lượng đáp án hợp lệ khác nhau** mà các cách trả về (để thấy chúng không bắt buộc giống nhau).

In [28]:
rng = random.Random(7910)
different = 0
total = 0
for _ in range(2000):
    m = rng.randint(1, 26)
    order = "".join(rng.sample(ascii_lowercase, m))
    s = "".join(rng.choice(ascii_lowercase) for _ in range(rng.randint(1, 200)))
    outs = {}
    for name, fn in METHODS.items():
        if name == "1-hoan-vi":
            continue
        outs[name] = fn(order, s)
        check_property(order, s, outs[name])
    total += 1
    if len(set(outs.values())) > 1:
        different += 1
print(f"2000 input ngẫu nhiên: OK. Có {different}/{total} input mà các cách cho các chuỗi khác nhau (đều hợp lệ).")

2000 input ngẫu nhiên: OK. Có 1775/2000 input mà các cách cho các chuỗi khác nhau (đều hợp lệ).


In [29]:
# Hiệu năng ở giới hạn: 20000 lần với s dài 200, order dài 26
rng = random.Random(1)
order = "".join(rng.sample(ascii_lowercase, 26))
s = "".join(rng.choice(ascii_lowercase) for _ in range(200))
for name in ["2-sort-key", "3-cmp", "4-dem", "5-doi-nhan", "nop"]:
    fn = METHODS[name]
    t0 = time.perf_counter()
    for _ in range(20000):
        fn(order, s)
    print(f"{name:11} {(time.perf_counter() - t0) * 1000:8.1f} ms / 20000 lần")

2-sort-key     454.0 ms / 20000 lần


3-cmp         2500.0 ms / 20000 lần
4-dem          184.1 ms / 20000 lần


5-doi-nhan     373.4 ms / 20000 lần
nop            185.0 ms / 20000 lần


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Làm rơi các chữ cái không có trong `order`

**Code sai:** chỉ ghép các chữ cái của `order`.

In [30]:
def wrong_drop_others(order, s):
    cnt = Counter(s)
    return "".join(c * cnt[c] for c in order)           # SAI: chữ cái ngoài order biến mất

order, s = "cba", "abcd"
got = wrong_drop_others(order, s)
print("sai  :", repr(got), "hợp lệ?", is_valid(order, s, got), "(thiếu 'd': không còn là hoán vị của s)")
print("đúng :", repr(Solution().customSortString(order, s)), "hợp lệ?", is_valid(order, s, Solution().customSortString(order, s)))

sai  : 'cba' hợp lệ? False (thiếu 'd': không còn là hoán vị của s)
đúng : 'cbad' hợp lệ? True


**Vì sao sai:** kết quả phải là **hoán vị của toàn bộ `s`**. Chữ `d` không có trong `order` vẫn phải xuất hiện (ở vị trí tùy ý).

**Code đúng:** sau vòng qua `order`, thêm các chữ cái còn lại.

### Lỗi 2: Dùng `set` làm mất chữ cái lặp

In [31]:
def wrong_set(order, s):
    return "".join(c for c in order if c in set(s)) + "".join(c for c in set(s) if c not in order)   # SAI

order, s = "cba", "aabbccd"
got = wrong_set(order, s)
print("sai  :", repr(got), "hợp lệ?", is_valid(order, s, got), "(mất các bản sao)")
print("đúng :", repr(Solution().customSortString(order, s)))

sai  : 'cbad' hợp lệ? False (mất các bản sao)
đúng : 'ccbbaad'


**Vì sao sai:** `set` bỏ trùng lặp; chuỗi phải giữ **đủ số lần** xuất hiện của mỗi chữ cái. Dùng `Counter` (hoặc `s.count(c)`) để biết số lần.

**Code đúng:** `c * cnt[c]` cho mỗi chữ cái.

### Lỗi 3: `order.index(c)` với chữ cái vắng mặt

In [32]:
def wrong_index(order, s):
    return "".join(sorted(s, key=lambda c: order.index(c)))   # SAI: 'd' không có trong order

try:
    wrong_index("cba", "abcd")
except ValueError as e:
    print("sai  : ValueError:", e)
print("đúng :", repr(Solution2().customSortString("cba", "abcd")), "(dùng rank.get(c, len(order)))")

sai  : ValueError: substring not found
đúng : 'cbad' (dùng rank.get(c, len(order)))


**Vì sao sai:** `str.index` báo `ValueError` khi không tìm thấy. Chữ cái vắng mặt phải có khóa mặc định.

**Code đúng:** `rank.get(c, len(order))` (hoặc `order.find(c)` rồi xử lý `-1` cẩn thận: `-1` sẽ đưa chữ vắng mặt lên **đầu**, cũng hợp lệ).

### Lỗi 4: Đếm trùng - xuất chữ cái thuộc `order` hai lần

In [33]:
def wrong_double(order, s):
    cnt = Counter(s)
    res = [c * cnt[c] for c in order]
    res += [c * k for c, k in cnt.items()]                # SAI: quên loại các chữ cái của order đã xuất
    return "".join(res)

order, s = "cba", "abcd"
got = wrong_double(order, s)
print("sai  :", repr(got), "hợp lệ?", is_valid(order, s, got), "(dài gấp đôi phần thuộc order)")
print("đúng :", repr(Solution().customSortString(order, s)))

sai  : 'cbaabcd' hợp lệ? False (dài gấp đôi phần thuộc order)
đúng : 'cbad'


**Vì sao sai:** vòng thứ hai duyệt mọi chữ cái của `s`, kể cả những chữ đã được xuất ở vòng thứ nhất. Cần `pop` (hoặc `del`) chữ cái ngay sau khi xuất, hoặc lọc `c not in order` ở vòng thứ hai.

**Code đúng:** `cnt.pop(c)` ở vòng đầu, hoặc `if c not in order` ở vòng sau.

### Lỗi 5: Sắp xếp theo bảng chữ cái thường (bỏ qua `order`)

In [34]:
def wrong_alphabetical(order, s):
    return "".join(sorted(s))                              # SAI: không dùng order

order, s = "cba", "abcd"
got = wrong_alphabetical(order, s)
print("sai  :", repr(got), "hợp lệ?", is_valid(order, s, got), "(a, b, c đang đúng thứ tự xuôi, không phải c, b, a)")
print("đúng :", repr(Solution().customSortString(order, s)))

sai  : 'abcd' hợp lệ? False (a, b, c đang đúng thứ tự xuôi, không phải c, b, a)
đúng : 'cbad'


**Vì sao sai:** thứ tự mong muốn là thứ tự trong `order`, không phải thứ tự bảng chữ cái. Mọi so sánh phải đi qua **thứ hạng trong `order`**.

**Code đúng:** `sorted(s, key=rank...)` hoặc đếm rồi ghép theo `order`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Nhận dạng:** "sắp xếp theo thứ tự tùy chỉnh" -> gán mỗi phần tử một **thứ hạng** rồi sắp xếp theo thứ hạng; nếu miền giá trị nhỏ (26 chữ cái) -> **đếm** (counting sort) thay vì sắp xếp.
- **Nhiều đáp án đúng** -> test bằng **hàm kiểm tra** (`is_valid`) và tính chất bất biến (chiếu lên các chữ cái bị ràng buộc cho đáp án duy nhất), đừng so chuỗi cứng.
- Chữ cái ngoài `order` -> đặt cuối (hoặc đầu), bằng khóa mặc định `len(order)`.
- Ghép chuỗi bằng `"".join(list)` thay vì `+=` trong vòng lặp lớn.
- `Counter.pop(c)` vừa lấy số lần vừa xóa: tránh đếm trùng.
- Cần giữ **thứ tự ban đầu** của các chữ cái không bị ràng buộc: dùng `sorted` (ổn định) với khóa mặc định giống nhau.
- Bài họ hàng: LeetCode 953 (Verifying an Alien Dictionary), 451 (Sort Characters By Frequency), 1366 (Rank Teams by Votes), 242 (Valid Anagram), 75 (Sort Colors), 1122 (Relative Sort Array), 49 (Group Anagrams).

## 15. Biến thể

### 15.1. Đáp án xác định: chữ cái ngoài `order` giữ nguyên thứ tự ban đầu, đặt ở cuối

Dùng `sorted` ổn định với khóa `rank.get(c, len(order))` cho ra đúng điều này: các chữ cái ngoài `order` có cùng khóa nên giữ thứ tự xuất hiện trong `s`.

In [35]:
def custom_sort_stable(order, s):
    rank = {c: i for i, c in enumerate(order)}
    return "".join(sorted(s, key=lambda c: rank.get(c, len(order))))

print(custom_sort_stable("cba", "xaybzc"))        # x, y, z giữ thứ tự gốc ở cuối
assert custom_sort_stable("cba", "xaybzc") == "cbaxyz"
assert custom_sort_stable("cba", "xaybzc").endswith("xyz")

cbaxyz


### 15.2. Verifying an Alien Dictionary (LeetCode 953)

Cho `words` và một thứ tự bảng chữ cái `order`, kiểm tra `words` có được sắp xếp theo thứ tự đó không. Ta tái dùng thứ hạng: đổi mỗi từ thành dãy thứ hạng rồi so sánh dãy.

In [36]:
def is_alien_sorted(words, order):
    rank = {c: i for i, c in enumerate(order)}
    keys = [[rank[c] for c in w] for w in words]
    return all(keys[i] <= keys[i + 1] for i in range(len(keys) - 1))

print(is_alien_sorted(["hello", "leetcode"], "hlabcdefgijkmnopqrstuvwxyz"))
print(is_alien_sorted(["word", "world", "row"], "worldabcefghijkmnpqstuvxyz"))
print(is_alien_sorted(["apple", "app"], "abcdefghijklmnopqrstuvwxyz"))

True
False
False


### 15.3. Sắp xếp theo tần suất (LeetCode 451)

Cùng công cụ `Counter` nhưng khóa là **tần suất giảm dần**: chữ cái xuất hiện nhiều nhất đứng trước, các bản sao đứng cạnh nhau.

In [37]:
def frequency_sort(s):
    cnt = Counter(s)
    return "".join(c * k for c, k in sorted(cnt.items(), key=lambda t: (-t[1], t[0])))

print(frequency_sort("tree"), frequency_sort("cccaaa"), frequency_sort("Aabb"))

eert aaaccc bbAa


### 15.4. Sắp xếp một mảng theo mảng thứ tự cho trước (LeetCode 1122)

`arr1` sắp theo thứ tự của `arr2` (các phần tử không có trong `arr2` xếp tăng dần ở cuối). Cũng là "đếm rồi ghép": số nhỏ nên dùng `Counter`.

In [38]:
def relative_sort(arr1, arr2):
    cnt = Counter(arr1)
    res = []
    for x in arr2:
        res += [x] * cnt.pop(x, 0)
    for x in sorted(cnt):
        res += [x] * cnt[x]
    return res

print(relative_sort([2, 3, 1, 3, 2, 4, 6, 7, 9, 2, 19], [2, 1, 4, 3, 9, 6]))

[2, 2, 2, 1, 4, 3, 3, 9, 6, 7, 19]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi hoán vị | kiểm tra từng hoán vị bằng `is_valid` | `O(n! * n)` | `O(n)` | chỉ để hiểu đề / sinh tập đáp án cho test nhỏ |
| 2. `sorted` theo khóa | khóa = thứ hạng trong `order` | `O(n log n)` | `O(n)` | viết nhanh, ổn định |
| 3. `sorted` theo hàm so sánh | luật so sánh tùy biến | `O(n log n)` | `O(n)` | luật phức tạp không viết gọn thành khóa |
| 4. Đếm rồi ghép | xuất `c * cnt[c]` theo `order` | `O(n + m)` | `O(n)` | **nộp bài** (tuyến tính) |
| 5. Đổi nhãn + `sorted` | đưa thứ tự tùy chỉnh về `a..z` | `O(n log n)` | `O(n)` | học kỹ thuật relabeling |

Kết luận: nộp **cách 4**. Cách 2 là phương án ngắn nhất đáng nhớ; các cách 3, 5 mở rộng góc nhìn (so sánh tùy biến, đổi nhãn).

## 17. 5 bài tập vận dụng (dễ → khó)

Cell **TODO** chỉ gán `None` hoặc định nghĩa hàm rỗng, **không được gọi** khi chạy notebook. Đáp án ở mục 19.

**Bài 1 (đoán output).** Dự đoán đầu ra rồi chạy (vì có nhiều đáp án đúng, hãy suy nghĩ "chuỗi nào hợp lệ" thay vì chuỗi chính xác):

In [39]:
res1 = Solution().customSortString("ba", "aabbb")
res2 = Solution().customSortString("xy", "yxz")
print(res1, is_valid("ba", "aabbb", res1))
print(res2, is_valid("xy", "yxz", res2))

bbbaa True
xyz True


**Bài 2 (tính tay).** `order = "kqep"`, `s = "pekeq"`. Hãy viết chuỗi kết quả theo cách đếm-rồi-ghép (không còn chữ cái nào ngoài `order`, nên đáp án là **duy nhất**) và gán vào `ans2` (đang là `None`).

In [40]:
ans2 = None      # TODO: điền một chuỗi

**Bài 3 (điền TODO).** Viết `rank_map(order)` trả về từ điển `{chữ cái: vị trí}`. Ví dụ `rank_map("cba")` -> `{"c": 0, "b": 1, "a": 2}`.

In [41]:
def rank_map(order):
    # TODO: dùng enumerate
    pass

**Bài 4 (tự viết lại).** Không nhìn mục 7, tự viết `my_custom_sort(order, s)` theo cách 4 (đếm rồi ghép theo `order`, sau đó chữ cái còn lại).

In [42]:
def my_custom_sort(order, s):
    # TODO
    pass

**Bài 5 (mini challenge).** (a) Viết `is_valid_mine(order, s, result)` tự kiểm tra tính hợp lệ (độc lập với `is_valid` ở mục 3.1). (b) Viết `count_valid(order, s)` đếm **số đáp án hợp lệ khác nhau** cho `s` ngắn bằng vét cạn hoán vị; ví dụ `count_valid("cba", "abcd")` = `4`.

In [43]:
def is_valid_mine(order, s, result):
    # TODO
    pass

def count_valid(order, s):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):**
- Bài 1: `order = "ba"` nghĩa là `b` đứng **trước** `a`; với `order = "xy"` thì `x` trước `y`, còn `z` tự do.
- Bài 2: đếm `k, q, e, p`; theo `order`: `k`, `q`, `e`, `p`.
- Bài 3: `enumerate(order)` cho `(chỉ số, chữ cái)`.
- Bài 4: dùng `Counter`, vòng lặp qua `order`, rồi các chữ còn lại.
- Bài 5: nhớ hai điều kiện: hoán vị của `s` và dãy thứ hạng không giảm.

**Mức 2 (vừa):**
- Bài 1: dòng 1 -> `"bbbaa"`; dòng 2: `x` trước `y`, còn `z` ở vị trí tùy ý.
- Bài 2: `e` xuất hiện hai lần -> `"ee"`.
- Bài 3: `{c: i for i, c in enumerate(order)}`.
- Bài 4: `res.append(c * cnt.pop(c))` nếu `c in cnt`.
- Bài 5(b): `{"".join(p) for p in permutations(s)}` rồi lọc bằng hàm kiểm tra.

**Mức 3 (gần đáp án):**
- Bài 2: `"kqeep"`.
- Bài 5(a): `sorted(result) == sorted(s)` và `ranks = [rank[c] for c in result if c in rank]` rồi `all(ranks[i] <= ranks[i + 1] ...)`.
- Bài 5(b): `len([x for x in set(...) if valid(x)])`.

## 19. Đáp án tham khảo

Chạy kiểm tra bằng `assert`.

In [44]:
# Bài 1: res1 = "bbbaa" (duy nhất vì chỉ có a, b); res2 có x trước y, z tùy ý
assert res1 == "bbbaa" and is_valid("ba", "aabbb", res1)
assert is_valid("xy", "yxz", res2) and res2.index("x") < res2.index("y")

# Bài 2: k, q, ee, p
ans2 = "kqeep"
assert ans2 == Solution().customSortString("kqep", "pekeq")
assert all(m("kqep", "pekeq") == ans2 for m in METHODS.values())        # đáp án duy nhất: mọi cách trùng
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [45]:
# Bài 3
def rank_map(order):
    return {c: i for i, c in enumerate(order)}

assert rank_map("cba") == {"c": 0, "b": 1, "a": 2}
assert rank_map("a") == {"a": 0}
assert rank_map("") == {}
print("Bài 3 OK")

Bài 3 OK


In [46]:
# Bài 4
def my_custom_sort(order, s):
    cnt = Counter(s)
    res = []
    for c in order:
        if c in cnt:
            res.append(c * cnt.pop(c))
    for c, k in cnt.items():
        res.append(c * k)
    return "".join(res)

rng = random.Random(4)
for _ in range(300):
    m = rng.randint(1, 10)
    order = "".join(rng.sample(ascii_lowercase, m))
    s = "".join(rng.choice(ascii_lowercase[:14]) for _ in range(rng.randint(1, 40)))
    check_property(order, s, my_custom_sort(order, s))
print("Bài 4 OK")

Bài 4 OK


In [47]:
# Bài 5
def is_valid_mine(order, s, result):
    if Counter(result) != Counter(s):
        return False
    pos = {c: i for i, c in enumerate(order)}
    ranks = [pos[c] for c in result if c in pos]
    return all(ranks[i] <= ranks[i + 1] for i in range(len(ranks) - 1))

def count_valid(order, s):
    return len({"".join(p) for p in permutations(s) if is_valid_mine(order, s, "".join(p))})

assert count_valid("cba", "abcd") == 4
assert count_valid("ab", "ab") == 1
assert count_valid("xyz", "abc") == 6                      # không chữ nào thuộc order: mọi hoán vị đều hợp lệ
rng = random.Random(5)
for _ in range(300):
    alphabet = rng.choice(["abc", "abcd", "abcde"])
    order = "".join(rng.sample(alphabet, rng.randint(1, len(alphabet))))
    s = "".join(rng.choice(alphabet) for _ in range(rng.randint(1, 5)))
    res = "".join(rng.sample(s, len(s)))
    assert is_valid_mine(order, s, res) == is_valid(order, s, res)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (đếm rồi ghép):**

```python
from collections import Counter

class Solution:
    def customSortString(self, order: str, s: str) -> str:
        cnt = Counter(s)
        res = []
        for c in order:                     # theo đúng thứ tự order
            if c in cnt:
                res.append(c * cnt.pop(c))  # lấy hết các bản sao, bỏ khỏi bộ đếm
        for c, k in cnt.items():            # chữ cái ngoài order: đặt cuối
            res.append(c * k)
        return "".join(res)
```

**Template ngắn (sắp xếp theo khóa):**

```python
rank = {c: i for i, c in enumerate(order)}
return "".join(sorted(s, key=lambda c: rank.get(c, len(order))))
```

**Checklist trước khi submit:**

- [ ] Kết quả là **hoán vị của toàn bộ `s`** (không rơi chữ cái ngoài `order`, không mất bản sao)?
- [ ] Mỗi chữ cái trong `order` được xuất **đúng một lần** theo `cnt[c]` (đã `pop`/lọc để không đếm trùng)?
- [ ] Chữ cái vắng mặt trong `order` có khóa mặc định (không dùng `order.index`)?
- [ ] Khi test: dùng **hàm kiểm tra tính hợp lệ**, không so chuỗi cứng?
- [ ] Đã thử: không chữ nào thuộc `order`, chữ cái lặp nhiều, `order` có chữ không xuất hiện trong `s`, `order` đủ 26 chữ?